# Ön Çalışma: Şekil 1 ve Şekil 2'yi kendin üret

Bu defter, başvurudaki **Şekil 1** (WASP-12 b O−C diyagramı) ve **Şekil 2**'yi (dört hedef) baştan üretir.
Hücreleri **yukarıdan aşağı, sırayla** çalıştır (hücreye tıkla → **Shift + Enter**). Her hücrenin üstünde ne yaptığı ve
ekranda ne görmen gerektiği yazıyor.

Kullanılan her şey herkese açık:
- **Literatür zamanları:** Ivshina & Winn (2022) tablosu (VizieR)
- **Wang vd. (2024) TESS zamanları:** VizieR
- **Yeni TESS ışık eğrileri:** NASA MAST arşivi

Süre: yaklaşık 20–30 dakika (çoğu bekleme).

## Adım 1 — Gerekli Python paketlerini kur
`astropy` (zaman ve koordinat), `astroquery` (arşivlerden veri indirme), `batman` (geçiş modeli), `emcee` (hata hesabı için MCMC).
Bitince ekranda hata görmemelisin (uyarılar önemli değil).

In [ ]:
!pip install -q astropy astroquery batman-package emcee

## Adım 2 — İki kod dosyasını oluştur
Aşağıdaki iki hücre, kodları Colab'ın klasörüne `oc_diyagrami.py` ve `tess_zaman.py` adıyla kaydeder. Çalıştır, sadece
"Writing ..." yazısı görmelisin. İstersen kodu okuyabilirsin; her fonksiyonun ne yaptığı içinde açıklanıyor.

- `oc_diyagrami.py`: Literatür ve Wang vd. zamanlarını indirir, epokları hesaplar, hata çubuklarını gerçekçi hale getirir,
  doğru ve parabol uydurur, dP/dt ve ΔBIC'i yazar, O−C şeklini çizer.
- `tess_zaman.py`: TESS ışık eğrisini indirir, her geçişin orta zamanını (O) ölçer.

In [ ]:
%%writefile oc_diyagrami.py
"""O-C diyagrami: Ivshina & Winn (2022) + Wang vd. (2024) gecis zamanlari.
Kullanim:  python oc_diyagrami.py "WASP-12 b" WASP-012 [tess_WASP-12.csv ...]
"""
import sys, io, urllib.parse, urllib.request
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VIZIER = "https://vizier.cfa.harvard.edu/viz-bin/asu-tsv"   # Harvard aynasi; olmazsa vizier.cds.unistra.fr
YIL_S = 365.25 * 86400.0                                    # bir yil, saniye
EK_DOSYALAR = []
BELIRSIZ_BJD_DAHIL = True                                   # False yapip sonucun degisip degismedigine bak


def vizier(tablo, **filtre):
    """VizieR tablosunu indir; satir listesi (sozluk) dondur."""
    p = {"-source": tablo, "-out.max": "99999", "-out.all": ""}
    p.update(filtre)
    url = VIZIER + "?" + urllib.parse.urlencode(p)
    metin = urllib.request.urlopen(url, timeout=120).read().decode()
    satir = [s for s in metin.splitlines() if s and not s.startswith("#")]
    bas = [h.strip() for h in satir[0].split("\t")]
    return [dict(zip(bas, [x.strip() for x in s.split("\t")])) for s in satir[3:]]


def zamanlari_topla(gezegen, iw_adi):
    """O: gozlenen orta gecis zamanlari (BJD_TDB), hatalari ve kaynak etiketi."""
    t, s, k = [], [], []
    for r in vizier("J/ApJS/259/62/table4", Sys=iw_adi):        # literatur + I&W'nin TESS zamanlari
        sistem = r["n_TTmid"]
        if sistem not in ("BJD_TDB", "BJD_TT"):                  # TT ile TDB farki < 2 ms: ihmal edilir
            if sistem == "BJD" and BELIRSIZ_BJD_DAHIL:             # "BJD" = UTC mi TDB mi belli degil
                print("UYARI: zaman sistemi belirsiz (BJD), makaleden kontrol et:", r["TTmid"], r["Ref"])
            else:
                print("ATLANDI: zaman sistemi", sistem, r["TTmid"], r["Ref"])
                continue
        t.append(float(r["TTmid"])); s.append(float(r["e_TTmid"]))
        k.append("TESS" if r["Ref"] == "This work" else "Literatur")
    for r in vizier("J/ApJS/270/14/table1", Name=gezegen):      # Wang vd. TESS zamanlari
        tc = float(r["Tc"])
        if np.min(np.abs(np.array(t) - tc)) < 0.01:              # ayni gecis zaten var: tekrar sayma
            continue
        t.append(tc); s.append(float(r["e_Tc"])); k.append("TESS")
    for dosya in EK_DOSYALAR:                                     # kendi olctugun TESS zamanlari
        for _, tc, h in np.atleast_2d(np.loadtxt(dosya, delimiter=",")):
            if np.min(np.abs(np.array(t) - tc)) < 0.01:
                continue
            t.append(tc); s.append(h); k.append("TESS (bu proje)")
    i = np.argsort(t)
    return np.array(t)[i], np.array(s)[i], np.array(k)[i]


def uydur(E, t, s, derece):
    """Agirlikli en kucuk kareler. derece=1 dogrusal, 2 kuadratik. (katsayilar, kovaryans, chi2)."""
    X = np.vander(E, derece + 1, increasing=True)              # [1, E, E^2]
    A, b = X / s[:, None], t / s
    p, *_ = np.linalg.lstsq(A, b, rcond=None)
    kov = np.linalg.inv(A.T @ A)
    chi2 = float(np.sum((b - A @ p) ** 2))
    return p, kov, chi2


def analiz(E, t, s):
    """Dogrusal vs kuadratik: dP/dt (ms/yil), hatasi, Delta BIC."""
    n = len(t)
    p1, _, c1 = uydur(E, t, s, 1)
    p2, k2, c2 = uydur(E, t, s, 2)
    P = p2[1]
    pdot = 2 * p2[2] / P * YIL_S * 1e3                          # dP/dt = (1/P) dP/dN, dP/dN = 2c
    pdot_h = 2 * np.sqrt(k2[2, 2]) / P * YIL_S * 1e3
    dbic = (c1 + 2 * np.log(n)) - (c2 + 3 * np.log(n))
    return dict(p1=p1, p2=p2, pdot=pdot, pdot_h=pdot_h, dbic=dbic, chi2_1=c1, chi2_2=c2, n=n)


def sigma_kirp(E, t, s, n_sigma=5):
    """Wang vd. gibi: kuadratik artiklarda n-sigma disindakileri yinelemeli at."""
    tut = np.ones(len(t), bool)
    while True:
        p, _, _ = uydur(E[tut], t[tut], s[tut], 2)
        r = t - np.polyval(p[::-1], E)
        m, sd = r[tut].mean(), r[tut].std()
        yeni = np.abs(r - m) < n_sigma * sd
        if (yeni == tut).all():
            return tut
        tut = yeni


def hata_olcekle(E, t, s, k, tur=5):
    """Hata cubuklarini veri grubu (literatur / TESS) basina, kuadratik model artiklarinin
    indirgenmis ki-karesi 1 olacak sekilde buyutur (hicbir zaman kucultmez)."""
    s = s.copy()
    gruplar = [k == "Literatur", k != "Literatur"]
    for _ in range(tur):
        p, _, _ = uydur(E, t, s, 2)
        r = (t - np.polyval(p[::-1], E)) / s
        for g in gruplar:
            if g.sum() > 3:
                s[g] *= np.sqrt(max(np.sum(r[g] ** 2) / (g.sum() - 1), 1.0))
    return s


def loocv(E, t, s):
    """Her noktayi tek tek cikar, yeniden uydur; 3-sigma ve DBIC>10 kosulu hep saglaniyor mu?"""
    sonuc = []
    for i in range(len(t)):
        m = np.arange(len(t)) != i
        a = analiz(E[m], t[m], s[m])
        sonuc.append(abs(a["pdot"]) / a["pdot_h"] >= 3 and a["dbic"] > 10)
    return np.array(sonuc)


if __name__ == "__main__":
    gezegen, iw_adi, EK_DOSYALAR = sys.argv[1], sys.argv[2], sys.argv[3:]
    t, s, k = zamanlari_topla(gezegen, iw_adi)
    # C icin baslangic efemerisi: Wang Tablo 2 (yoksa NASA Exoplanet Archive'dan T0, P)
    ef = [r for r in vizier("J/ApJS/270/14/table2", Name=gezegen)][0]
    T0, P = float(ef["T0"]), float(ef["P0"])
    E = np.round((t - T0) / P)
    E = E - np.round(np.median(E))                              # epoklari ortala: sayisal kararlilik
    tut = sigma_kirp(E, t, s, 5)
    E, t, s, k = E[tut], t[tut], s[tut], k[tut]
    s = hata_olcekle(E, t, s, k)                               # gercekci hata cubuklari
    a = analiz(E, t, s)
    gecti = loocv(E, t, s)
    print(f"{gezegen}: n={a['n']} (TESS {np.sum(k!='Literatur')}), dP/dt = {a['pdot']:.2f} +- {a['pdot_h']:.2f} ms/yil, "
          f"DBIC = {a['dbic']:.1f}, LOOCV: {np.sum(~gecti)} noktada kosul bozuluyor -> {'GECTI' if gecti.all() else 'KALDI'}")

    # O-C: dogrusal efemerise gore (C = T0 + P*E)
    C = a["p1"][0] + a["p1"][1] * E
    oc_dk = (t - C) * 1440
    ee = np.linspace(E.min(), E.max(), 400)
    kuad = (np.polyval(a["p2"][::-1], ee) - (a["p1"][0] + a["p1"][1] * ee)) * 1440
    fig, ax = plt.subplots(figsize=(7, 4))
    for etiket, renk in [("Literatur", "tab:orange"), ("TESS", "tab:green"), ("TESS (bu proje)", "tab:red")]:
        m = k == etiket
        if not m.any():
            continue
        ax.errorbar(E[m], oc_dk[m], s[m] * 1440, fmt="o", ms=3, color=renk, label=etiket, alpha=.8)
    ax.plot(ee, kuad, "b-", label=f"kuadratik: dP/dt = {a['pdot']:.1f} ± {a['pdot_h']:.1f} ms/yıl")
    ax.axhline(0, color="k", lw=.8, ls="--", label="doğrusal (sabit dönem)")
    ax.set_xlabel("Epok (E)"); ax.set_ylabel("O − C (dakika)")
    ax.set_title(f"{gezegen}  (ΔBIC = {a['dbic']:.1f}, n = {a['n']})")
    ax.legend(fontsize=8); fig.tight_layout()
    ad = gezegen.replace(" ", "")
    fig.savefig(f"OC_{ad}.png", dpi=200)
    np.savetxt(f"zamanlar_{ad}.csv", np.column_stack([E, t, s, oc_dk]), delimiter=",",
               header="E,T_gozlenen_BJD_TDB,hata_gun,OC_dakika", fmt=["%d", "%.6f", "%.6f", "%.3f"])


In [ ]:
%%writefile tess_zaman.py
"""TESS isik egrisinden orta gecis zamanlari (O) olcer.
Kullanim:  python tess_zaman.py WASP-12 97.6366450 29.6722662 1.09141890 2457607.51930 71 72
           (hedef, RA, Dec, P [gun], T0 [BJD_TDB], sektor listesi)
"""
import sys
import numpy as np
import batman, emcee
from scipy.optimize import least_squares
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astroquery.mast import Observations

BTJD0 = 2457000.0                         # TESS zamani: BTJD = BJD_TDB - 2457000


def indir(ra, dec, sektorler):
    """MAST'tan SPOC 120 s isik egrisi (*_lc.fits) dosyalarini indirir."""
    obs = Observations.query_criteria(coordinates=SkyCoord(ra, dec, unit="deg"), radius="0.003 deg",
                                      obs_collection="TESS", dataproduct_type="timeseries",
                                      provenance_name="SPOC")
    obs = obs[np.isin(obs["sequence_number"], sektorler)]
    urun = Observations.get_product_list(obs)
    urun = urun[[str(f).endswith("_lc.fits") for f in urun["productFilename"]]]
    liste = Observations.download_products(urun, download_dir="veri_ham")
    return sorted(str(p) for p in liste["Local Path"])        # yalnizca BU hedefin dosyalari


def oku(dosya):
    """TIME (BTJD), PDCSAP_FLUX; kalite bayragi 0 olmayan ve NaN noktalari atar, 1'e normalize eder."""
    d = fits.getdata(dosya, 1)
    m = (d["QUALITY"] == 0) & np.isfinite(d["TIME"]) & np.isfinite(d["PDCSAP_FLUX"])
    t, f, e = d["TIME"][m], d["PDCSAP_FLUX"][m], d["PDCSAP_FLUX_ERR"][m]
    return t + BTJD0, f / np.nanmedian(f), e / np.nanmedian(f)


def model(t, tc, P, rp, a, inc, u1=0.4, u2=0.2):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc, p.ecc, p.w = tc, P, rp, a, inc, 0.0, 90.0
    p.u, p.limb_dark = [u1, u2], "quadratic"
    return batman.TransitModel(p, t).light_curve(p)


def efemeris_merkezle(t, f, T0, P):
    """Eski bir T0 TESS'e kadar dakikalarca kaymis olabilir: katlanmis isik egrisinde
    en derin noktayi kutu taramasiyla bul ve T0'i oraya tasi (BLS benzeri basit tarama)."""
    n_ort = np.round((np.median(t) - T0) / P)
    T0 = T0 + n_ort * P                                       # T0'i TESS verisinin ortasina tasi
    kaymalar = np.arange(-0.15, 0.15, 0.001)
    derinlik = []
    for dt in kaymalar:
        x = ((t - T0 - dt) / P + 0.5) % 1 - 0.5
        derinlik.append(np.mean(f[np.abs(x * P) < 0.03]))     # +-43 dk kutu
    return T0 + kaymalar[int(np.argmin(derinlik))]


def sekil_uydur(t, f, e, T0, P):
    """Tum gecisleri katlayip (phase-fold) gecis seklini (rp, a, inc) bir kez belirler."""
    faz = ((t - T0) / P + 0.5) % 1 - 0.5                     # -0.5..0.5, gecis 0'da
    m = np.abs(faz) < 0.1
    x = faz[m] * P
    r = least_squares(lambda q: (f[m] - q[3] * model(x, 0.0, P, *q[:3])) / e[m],
                      [0.1, 4.0, 85.0, 1.0], bounds=([0.01, 1.5, 70, 0.9], [0.3, 20, 90, 1.1]))
    return r.x[:3]


def tek_gecis(t, f, e, tc0, P, sekil, mcmc=True):
    """Bir gecis: serbest parametreler tc + dogrusal taban (c0, c1). Sonuc: tc, hata (gun)."""
    def m(q):
        return model(t, q[0], P, *sekil) * (q[1] + q[2] * (t - tc0))
    ls = least_squares(lambda q: (f - m(q)) / e, [tc0, 1.0, 0.0])
    if not mcmc:
        J = ls.jac; kov = np.linalg.inv(J.T @ J)
        return ls.x[0], np.sqrt(kov[0, 0]), f - m(ls.x)
    def lnp(q):
        if abs(q[0] - tc0) > 0.05:
            return -np.inf
        return -0.5 * np.sum(((f - m(q)) / e) ** 2)
    sm = emcee.EnsembleSampler(12, 3, lnp)
    sm.run_mcmc(ls.x + 1e-5 * np.random.randn(12, 3), 2000, progress=False)
    z = sm.get_chain(discard=500, flat=True)[:, 0]
    q16, q50, q84 = np.percentile(z, [16, 50, 84])
    return q50, 0.5 * (q84 - q16), f - m(ls.x)


if __name__ == "__main__":
    hedef, ra, dec, P, T0 = sys.argv[1], *map(float, sys.argv[2:6])
    sektorler = [int(x) for x in sys.argv[6:]]
    t_hep, f_hep, e_hep = [], [], []
    for d in indir(ra, dec, sektorler):
        t, f, e = oku(d); t_hep.append(t); f_hep.append(f); e_hep.append(e)
    t, f, e = map(np.concatenate, (t_hep, f_hep, e_hep))
    T0_eski = T0
    T0 = efemeris_merkezle(t, f, T0, P)
    for tur in (1, 2):                                        # 1. tur: hizli (en kucuk kareler); 2. tur: MCMC
        sekil = sekil_uydur(t, f, e, T0, P)
        print(f"Tur {tur}: Rp/R* = {sekil[0]:.4f}, a/R* = {sekil[1]:.2f}, i = {sekil[2]:.2f}")
        sure = P / np.pi / sekil[1]                          # kabaca gecis suresi (gun)
        satir = []
        for n in range(int(np.ceil((t.min() - T0) / P)), int(np.floor((t.max() - T0) / P)) + 1):
            tc0 = T0 + n * P
            w = np.abs(t - tc0) < 1.5 * sure                   # gecisin iki yani + taban
            giris = np.any((t > tc0 - 0.6 * sure) & (t < tc0 - 0.4 * sure))
            cikis = np.any((t > tc0 + 0.4 * sure) & (t < tc0 + 0.6 * sure))
            if w.sum() < 50 or not (giris and cikis):           # kismi gecis: kullanma
                continue
            tc, h, art = tek_gecis(t[w], f[w], e[w], tc0, P, sekil, mcmc=(tur == 2))
            satir.append((n, tc, h))
        n_, tc_, h_ = np.array(satir).T
        q = np.polyfit(n_, tc_, 1, w=1 / h_)                  # TESS zamanlarindan yerel efemeris
        P, T0 = q[0], q[1]
    for n, tc, h in satir:
        print(f"Tc = {tc:.6f} BJD_TDB  +- {h * 86400:5.1f} s")
    satir = [(np.round((tc - T0_eski) / P), tc, h) for _, tc, h in satir]   # epok: girilen T0'a gore
    np.savetxt(f"tess_{hedef}.csv", satir, delimiter=",", fmt=["%d", "%.6f", "%.6f"],
               header="epok,Tc_BJD_TDB,hata_gun")


## Adım 3 — Sadece hazır katalog verisiyle WASP-12 b
Bu hücre literatür + Wang vd. TESS zamanlarını VizieR'den indirir ve analiz eder (≈1 dakika).

**Görmen gereken:** `WASP-12 b: n=189 (TESS 82), dP/dt = -28.69 +- 1.03 ms/yil, DBIC = 770.5 ...`
Literatür değeri −29 ± 2 ms/yıl (Yee vd. 2020): uyuyor.

In [ ]:
!python oc_diyagrami.py "WASP-12 b" WASP-012

## Adım 4 — Ölçüm yöntemini doğrula (Wang vd.'nin ölçtüğü sektör 20)
Önce Wang vd.'nin **zaten ölçtüğü** bir TESS sektörünü (20) kendimiz ölçüyoruz; sonra onların sonucuyla karşılaştıracağız.
Bu, "bizim ölçümümüz doğru mu?" sorusunun cevabı. (≈3–5 dakika; MAST'tan dosya iner, her geçiş için MCMC çalışır.)

**Görmen gereken:** `Tur 2: Rp/R* ≈ 0.114, a/R* ≈ 3.1` ve ~20 satır `Tc = ... BJD_TDB +- ~45 s`.

In [ ]:
!python tess_zaman.py WASP-12 97.6366450 29.6722662 1.091418901 2457607.519305 20
!mv tess_WASP-12.csv dogrulama_sektor20.csv

Şimdi kendi ölçtüğün sektör-20 zamanlarını Wang vd.'nin aynı geçişleriyle karşılaştır.

**Görmen gereken:** `ortak gecis: 20 | ortalama fark ≈ -2 s | ortanca |fark|/sigma ≈ 0.4`
Yani sistematik kayma yok, farklar hata payının içinde → **ölçüm yöntemi doğrulandı.**

In [ ]:
import numpy as np, oc_diyagrami as oc
ben = np.loadtxt("dogrulama_sektor20.csv", delimiter=",")
w = oc.vizier("J/ApJS/270/14/table1", Name="WASP-12 b")
wt = np.array([float(x["Tc"]) for x in w]); ws = np.array([float(x["e_Tc"]) for x in w])
fark, sig = [], []
for _, tc, h in ben:
    j = np.argmin(abs(wt - tc))
    if abs(wt[j] - tc) < 0.02:
        fark.append((tc - wt[j]) * 86400); sig.append((tc - wt[j]) / np.hypot(h, ws[j]))
fark, sig = np.array(fark), np.array(sig)
print(f"ortak gecis: {len(fark)} | ortalama fark {fark.mean():+.1f} s | ortanca |fark|/sigma {np.median(abs(sig)):.2f}")

## Adım 5 — Wang vd. sonrası yeni TESS sektörleri (71 ve 72)
Wang vd.'nin kullanmadığı, 2023'te çekilmiş iki sektörü ölçüyoruz (≈5–8 dakika).

**Görmen gereken:** ~38 satır `Tc = ...`, sonunda `tess_WASP-12.csv` dosyası oluşur.

In [ ]:
!python tess_zaman.py WASP-12 97.6366450 29.6722662 1.091418901 2457607.519305 71 72

## Adım 6 — Şekil 1: katalog + yeni TESS zamanlarıyla O−C
**Görmen gereken:** `WASP-12 b: n=227 (TESS 120), dP/dt = -30.92 +- 0.90 ms/yil, DBIC = 1179.6`
ve `OC_WASP-12b.png` dosyası. Aşağıdaki hücre şekli ekranda da gösterir. **Bu dosya başvurudaki Şekil 1'dir.**

In [ ]:
!python oc_diyagrami.py "WASP-12 b" WASP-012 tess_WASP-12.csv
from IPython.display import Image
Image("OC_WASP-12b.png")

## Adım 7 — Diğer üç hedef (Şekil 2 için)
Her hedefin Wang sonrası TESS sektörlerini ölçüp O−C'yi çizer. Bu adım uzun sürer (toplam ≈10–15 dakika).

**Görmen gereken (yaklaşık):**
- `WASP-4 b: ... dP/dt = -4.83 +- 0.40` (75 yeni geçiş)
- `HAT-P-7 b: ... dP/dt = 30.24 +- 4.72` (40 yeni geçiş)
- `TrES-1 b: ... dP/dt = -4.51 +- 2.63` (21 yeni geçiş — anlamlı değişim yok)

In [ ]:
!python tess_zaman.py WASP-4 353.5629149 -42.0617794 1.338231 2455804.515679 36 69 96 103 105 106
!python tess_zaman.py HAT-P-7 292.2471900 47.9695429 2.204737 2454954.358572 74 75 78 81 82 86
!python tess_zaman.py TrES-1 286.0408755 36.6325360 3.03007 2453186.806341 74 78 80 81 86
!python oc_diyagrami.py "WASP-4 b" WASP-004 tess_WASP-4.csv
!python oc_diyagrami.py "HAT-P-7 b" HAT-P-07 tess_HAT-P-7.csv
!python oc_diyagrami.py "TrES-1 b" TrES-1 tess_TrES-1.csv

## Adım 8 — Şekil 2: dört paneli birleştir
**Bu dosya (`sekil2_dort_hedef_OC.png`) başvurudaki Şekil 2'dir.** Sol menüdeki klasör simgesinden PNG dosyalarını indir.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as im
fig, ax = plt.subplots(2, 2, figsize=(14, 8.2))
for a, ad in zip(ax.flat, ["WASP-12b", "WASP-4b", "HAT-P-7b", "TrES-1b"]):
    a.imshow(im.imread(f"OC_{ad}.png")); a.axis("off")
plt.tight_layout(); plt.savefig("sekil2_dort_hedef_OC.png", dpi=150); plt.show()